[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_15/SFM_ch15_networks/SFM_ch15_networks.ipynb)

# SFM_ch15_networks

Networks of 11 US and European banks: pairwise Granger-causality tests (lag 1, 5% level) on monthly returns in 36-month rolling windows and the degree of Granger causality (Billio, Getmansky, Lo and Pelizzon, 2012); the networks of 2007-2009 and 2017-2019; the Diebold-Yilmaz connectedness table of daily returns (generalized forecast error variance decomposition, H = 10 days, VAR order by BIC) and the total connectedness on 200-day rolling windows.

*Statistics of Financial Markets (SFM), Chapter 15: Systemic risk. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_15'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
import matplotlib.dates as mdates

import statsmodels.api as sm

from statsmodels.regression.quantile_regression import QuantReg

from scipy.special import gammaln

START, END = '2005-01-01', '2026-09-18'

BANKS = {'JPM': ('JPM.US', 'JPMorgan Chase', 'US'), 'BAC': ('BAC.US', 'Bank of America', 'US'), 'C': ('C.US', 'Citigroup', 'US'), 'GS': ('GS.US', 'Goldman Sachs', 'US'), 'MS': ('MS.US', 'Morgan Stanley', 'US'), 'WFC': ('WFC.US', 'Wells Fargo', 'US'), 'DBK': ('DBK.XETRA', 'Deutsche Bank', 'EU'), 'BNP': ('BNP.PA', 'BNP Paribas', 'EU'), 'SAN': ('SAN.MC', 'Santander', 'EU'), 'INGA': ('INGA.AS', 'ING', 'EU'), 'HSBA': ('HSBA.LSE', 'HSBC', 'EU'), 'TLV': ('TLV.RO', 'Banca Transilvania', 'RO'), 'BRD': ('BRD.RO', 'BRD', 'RO')}

REGIONS = {'US': ['JPM', 'BAC', 'C', 'GS', 'MS', 'WFC'], 'EU': ['DBK', 'BNP', 'SAN', 'INGA', 'HSBA'], 'RO': ['TLV', 'BRD']}

ALL = ['JPM', 'BAC', 'C', 'GS', 'MS', 'WFC', 'DBK', 'BNP', 'SAN', 'INGA', 'HSBA', 'TLV', 'BRD']

INTL = ['JPM', 'BAC', 'C', 'GS', 'MS', 'WFC', 'DBK', 'BNP', 'SAN', 'INGA', 'HSBA']

REGION_START = {'US': '2005-01-01', 'EU': '2005-01-01', 'RO': '2010-01-01'}

INDEX = {'US': 'sp500', 'EU': 'stoxx50', 'RO': 'bet'}

INDEX_NAME = {'sp500': 'S&P 500', 'stoxx50': 'Euro Stoxx 50', 'bet': 'BET'}

REG_COL = {'US': '#1A3A6E', 'EU': '#CD0000', 'RO': '#2E7D32'}

REG_LAB = {'US': 'US banks', 'EU': 'European banks', 'RO': 'Romanian banks (TLV, BRD, since 2010)'}

BAD_DAYS = {'TLV': ['2016-05-30', '2016-05-31']}

EVENTS = [('2008-09-15', 'Lehman'), ('2012-07-26', 'Draghi'), ('2020-03-16', 'COVID-19'), ('2023-03-10', 'SVB')]

EPISODES = {'Lehman 2008': ('2008-08-29', '2008-12-31'), 'Euro area 2011-2012': ('2011-06-30', '2012-09-28'), 'March 2020': ('2020-02-14', '2020-04-30'), 'March 2023': ('2023-02-28', '2023-04-28')}

ALPHA = 0.01

ALPHA_MES = 0.05

K_CAP = 0.08

B_BOOT, BLOCK = 200, 20

GC_WINDOW = 36

DY_H, DY_WINDOW, DY_STEP = 10, 200, 5

SEED = 2026

def bank_price(k, start=START, end=END):
    """Daily adjusted close of a bank (weekdays; no unchanged holiday-filled closes; BVB: no days without trades)."""
    sym, _, reg = BANKS[k]
    d = read_market(sym).loc[start:end]
    d = d[(d.index.dayofweek < 5) & (d['adjusted_close'] > 0)]
    if reg == 'RO' and 'volume' in d.columns:
        d = d[d['volume'] > 0]
    s = d['adjusted_close'].astype(float)
    return s[s.diff() != 0].rename(k)


def prices(keys, start=START, end=END):
    """Banks and indices (sp500, stoxx50, bet, vix) on their common trading days."""
    cols = [bank_price(k, start, end) if k in BANKS else load_close(k, start, end).rename(k) for k in keys]
    return pd.concat(cols, axis=1, join='inner').dropna()


def returns(keys, start=START, end=END):
    """Daily log returns in %: prices joined on common days first, then returns; known data errors removed."""
    R = 100 * np.log(prices(keys, start, end)).diff().dropna()
    for k, days in BAD_DAYS.items():
        if k in R.columns:
            R = R.drop(pd.to_datetime(days), errors='ignore')
    return R


def region_returns(reg, start=None, end=END):
    """The banks of a region and its equity index (S&P 500, Euro Stoxx 50 or BET), daily log returns in %."""
    return returns(REGIONS[reg] + [INDEX[reg]], max(start or START, REGION_START[reg]), end)


def bank_portfolio(R, members):
    """Equal-weighted bank portfolio (daily rebalancing): log return in % of the average simple return."""
    return 100 * np.log(1 + (np.exp(R[members] / 100) - 1).mean(axis=1))


def var_fit(Y, p):
    """VAR(p) by OLS: lag matrices A_1..A_p and the residual covariance matrix."""
    Y = np.asarray(Y, float)
    T, k = Y.shape
    X = np.hstack([np.ones((T - p, 1))] + [Y[p - l - 1:T - l - 1] for l in range(p)])
    B = np.linalg.lstsq(X, Y[p:], rcond=None)[0]
    E = Y[p:] - X @ B
    S = E.T @ E / (len(E) - X.shape[1])
    return [B[1 + l * k:1 + (l + 1) * k].T for l in range(p)], S


def var_bic(Y, pmax=4):
    """VAR order by the Bayesian information criterion."""
    Y = np.asarray(Y, float)
    T, k = Y.shape
    n = T - pmax
    best = None
    for p in range(1, pmax + 1):
        A, S = var_fit(Y[pmax - p:], p)
        bic = np.log(np.linalg.det(S * (n - 1 - p * k) / n)) + np.log(n) * p * k * k / n
        if best is None or bic < best[1]:
            best = (p, bic)
    return best[0]


def gfevd(A, S, H=DY_H):
    """Generalized forecast error variance decomposition (Pesaran and Shin, 1998), rows normalised to 1."""
    k = S.shape[0]
    Phi = [np.eye(k)]
    for h in range(1, H):
        Phi.append(sum(A[l] @ Phi[h - l - 1] for l in range(min(h, len(A)))))
    num, den = np.zeros((k, k)), np.zeros(k)
    for P in Phi:
        num += (P @ S) ** 2
        den += np.diag(P @ S @ P.T)
    th = num / np.diag(S)[None, :] / den[:, None]
    return th / th.sum(axis=1, keepdims=True)


def connectedness(theta, names):
    """Diebold-Yilmaz table: shares in %, FROM others, TO others, NET and the total connectedness index."""
    D = 100 * theta
    off = D - np.diag(np.diag(D))
    frm, to = off.sum(axis=1), off.sum(axis=0)
    return {'table': pd.DataFrame(D, index=names, columns=names), 'from': pd.Series(frm, index=names),
            'to': pd.Series(to, index=names), 'net': pd.Series(to - frm, index=names), 'total': float(off.sum() / len(names))}


def monthly_returns():
    """Monthly log returns in % of the 11 US and European banks (last common trading day of each month)."""
    P = prices(INTL)
    Pm = P.groupby(P.index.to_period('M')).last()
    return (100 * np.log(Pm).diff()).dropna()


def granger_p(y, x):
    """p-value of the test that x does not Granger-cause y: regression y_t = a + b y_{t-1} + c x_{t-1} + e_t, t test of c."""
    Y = y[1:]
    X = sm.add_constant(np.column_stack([y[:-1], x[:-1]]))
    return float(sm.OLS(Y, X).fit().pvalues[2])


def granger_network(Rm, level=0.05):
    """Matrix of significant Granger-causal links (row i causes column j) and the degree of Granger causality (DGC)."""
    ks = list(Rm.columns)
    A = np.zeros((len(ks), len(ks)), int)
    for i, a in enumerate(ks):
        for j, b in enumerate(ks):
            if i != j:
                A[i, j] = int(granger_p(Rm[b].values, Rm[a].values) < level)
    n = len(ks)
    return A, float(A.sum() / (n * (n - 1)))


def rolling_dgc(Rm=None, window=GC_WINDOW):
    """DGC on rolling windows of 36 months."""
    Rm = monthly_returns() if Rm is None else Rm
    out = {}
    for e in range(window, len(Rm) + 1):
        out[Rm.index[e - 1].to_timestamp('M')] = granger_network(Rm.iloc[e - window:e])[1]
    return pd.Series(out)


def fig_dgc(save=True):
    """Degree of Granger causality among the 11 US and European banks, 36-month windows; 5% line = links expected by chance."""
    Rm = monthly_returns()
    d = rolling_dgc(Rm)
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.plot(d.index, 100 * d, color=st.MainBlue, lw=1.5, label='degree of Granger causality (DGC), 36-month windows')
    ax.axhline(5, color=st.IDAred, ls='--', lw=1.2, label='5%: share of links expected by chance at the 5% level')
    for dd, lab in EVENTS:
        ax.axvline(pd.Timestamp(dd), color=st.Purple, ls=':', lw=1.0, label='_')
        ax.text(pd.Timestamp(dd), ax.get_ylim()[1], ' ' + lab, rotation=90, va='top', ha='right', fontsize=10, color='black')
    ax.set_ylabel('significant links (%)')
    st.legend_outside_bottom(ax, ncol=2, y=-0.13)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch15_dgc')
    return {'first': d.index[0].date().isoformat(), 'max': float(100 * d.max()), 'date_max': d.idxmax().date().isoformat(),
            'min': float(100 * d.min()), 'date_min': d.idxmin().date().isoformat(), 'mean': float(100 * d.mean()),
            'last': float(100 * d.iloc[-1]), 'n_months': len(Rm), 'start': str(Rm.index[0]),
            'd2009': float(100 * d.loc['2009-12-31']), 'd2019': float(100 * d.loc['2019-12-31'])}


def fig_granger_net(ends=('2009-12', '2019-12'), save=True):
    """The Granger-causality network of the 11 US and European banks in two 36-month windows (arrows: significant links at 5%)."""
    Rm = monthly_returns()
    ks = list(Rm.columns)
    ang = np.pi / 2 - 2 * np.pi * np.arange(len(ks)) / len(ks)
    pos = np.column_stack([np.cos(ang), np.sin(ang)])
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.0))
    out = {}
    for ax, e in zip(axes, ends):
        j = list(Rm.index.astype(str)).index(e) + 1
        A, dgc = granger_network(Rm.iloc[j - GC_WINDOW:j])
        for a in range(len(ks)):
            for b in range(len(ks)):
                if A[a, b]:
                    ax.annotate('', xy=pos[b] * 0.9, xytext=pos[a] * 0.9,
                                arrowprops=dict(arrowstyle='->', color=REG_COL[BANKS[ks[a]][2]], lw=0.9, alpha=0.8))
        for i, k in enumerate(ks):
            ax.scatter(*pos[i], s=380, color=REG_COL[BANKS[k][2]], zorder=3)
            ax.text(*(pos[i] * 1.22), k, ha='center', va='center', fontsize=10, color='black')
        a0 = Rm.index[j - GC_WINDOW].strftime('%b %Y')
        ax.set_title(f'{a0} – {Rm.index[j - 1].strftime("%b %Y")}: {A.sum()} links, DGC {100 * dgc:.0f}%', fontsize=11)
        ax.set_xlim(-1.35, 1.35)
        ax.set_ylim(-1.35, 1.35)
        ax.set_aspect('equal')
        ax.axis('off')
        out[e] = {'links': int(A.sum()), 'dgc': float(100 * dgc), 'out': {k: int(A[i].sum()) for i, k in enumerate(ks)}}
    h = [plt.Line2D([], [], color=REG_COL[r], marker='o', ls='-', ms=9) for r in ('US', 'EU')]
    fig.legend(h, [REG_LAB[r] + ' (arrows: links from this bank)' for r in ('US', 'EU')], loc='upper center',
               bbox_to_anchor=(0.5, 0.06), ncol=2, frameon=False)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch15_granger_net')
    return out


def dy_static():
    """Connectedness of the daily returns of the 11 US and European banks (common days since 2005)."""
    R = returns(INTL)
    p = var_bic(R.values)
    A, S = var_fit(R.values, p)
    c = connectedness(gfevd(A, S), list(R.columns))
    return c, p, len(R)


def fig_dy_table(c, save=True):
    """Heat map of the connectedness table (shares of forecast error variance, %); last row: TO others."""
    T = c['table']
    ks = list(T.index)
    M = np.vstack([T.values, c['to'].values])
    fig, ax = plt.subplots(figsize=(9.5, 6.2))
    ax.imshow(np.vstack([T.values, np.full(len(ks), np.nan)]), cmap='Blues', vmin=0, vmax=np.ceil(T.values.max() / 10) * 10)
    ax.set_xticks(range(len(ks)))
    ax.set_xticklabels(ks, fontsize=10.5)
    ax.xaxis.tick_top()
    ax.set_yticks(range(len(ks) + 1))
    ax.set_yticklabels([f'{k}  (FROM {c["from"][k]:.0f})' for k in ks] + ['TO others'], fontsize=10.5)
    for i in range(len(ks) + 1):
        for j in range(len(ks)):
            v = M[i, j]
            col = st.IDAred if i == len(ks) else ('white' if v > 0.55 * T.values.max() else 'black')
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=9.5, color=col)
    ax.set_xlabel(f'shock from (column) to (row), % of the forecast error variance\ntotal connectedness {c["total"]:.1f}%',
                  fontsize=11)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch15_dy_table')


def rolling_total(R=None, window=DY_WINDOW, step=DY_STEP, p=None):
    """Total connectedness on rolling windows of 200 days, moved by 5 days."""
    R = returns(INTL) if R is None else R
    p = var_bic(R.values) if p is None else p
    out = {}
    for e in range(window, len(R) + 1, step):
        A, S = var_fit(R.values[e - window:e], p)
        out[R.index[e - 1]] = connectedness(gfevd(A, S), list(R.columns))['total']
    return pd.Series(out)


def fig_dy_rolling(save=True):
    """Total connectedness of the 11 US and European banks, 200-day windows."""
    tot = rolling_total()
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.plot(tot.index, tot, color=st.MainBlue, lw=1.3, label='total connectedness, 200-day windows (%)')
    for dd, lab in EVENTS:
        ax.axvline(pd.Timestamp(dd), color=st.Purple, ls=':', lw=1.0, label='_')
        ax.text(pd.Timestamp(dd), ax.get_ylim()[1], ' ' + lab, rotation=90, va='top', ha='right', fontsize=10, color='black')
    ax.set_ylabel('total connectedness (%)')
    st.legend_outside_bottom(ax, ncol=1, y=-0.13)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch15_dy_rolling')
    return {'min': float(tot.min()), 'date_min': tot.idxmin().date().isoformat(), 'max': float(tot.max()),
            'date_max': tot.idxmax().date().isoformat(), 'last': float(tot.iloc[-1]), 'mean': float(tot.mean()),
            'y2017': float(tot.loc['2017'].mean()), 'y2008': float(tot.loc['2008-10-01':'2009-03-31'].mean())}

## Run

In [ ]:
print(fig_dgc())
print(fig_granger_net())
c, p, n = dy_static()
print(p, n, round(c['total'], 1))
print(c['table'].round(1))
fig_dy_table(c)
print(fig_dy_rolling())

![sfm_ch15_dgc](./sfm_ch15_dgc.png)

Output: `sfm_ch15_dgc.pdf`

![sfm_ch15_granger_net](./sfm_ch15_granger_net.png)

Output: `sfm_ch15_granger_net.pdf`

![sfm_ch15_dy_table](./sfm_ch15_dy_table.png)

Output: `sfm_ch15_dy_table.pdf`

![sfm_ch15_dy_rolling](./sfm_ch15_dy_rolling.png)

Output: `sfm_ch15_dy_rolling.pdf`